In [1]:
library(dplyr)
library(tidyr)
library(stringr)
library(vroom)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
# Read in data, for QTLs only keep summary stat variants

In [51]:
cred.set <- read.table('/nfs/lab/projects/nash_nafld_liver/GWAS/MVP_GWAS/NAFLD.TRANS.MVP.2021.credset.hg38.tsv', sep='\t', header=T)
dim(cred.set)
head(cred.set)

[1] 1084   18

,CS.Type,LEAD.SNP,CS.SNP,Chr,Position,EA,NEA,EAF,Beta,SE,P,N,PP,Nominated.Gene,Biological.Prior.Gene,chr.hg38,end.hg38,start.hg38
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>
1,Single SNP,rs2642438,rs2642438,1,220970028,A,G,0.274,-0.075,0.0074,6.65e-24,218595,96.86%,MTARC1,MTARC1,1,220796686,220796685
2,,rs6734238,rs6734238,2,113841030,G,A,0.407,-0.057,0.0064,4.94e-19,218595,99.04%,IL1RN,IL1RN,2,113083453,113083452
3,,rs138033684,rs138033684,6,71895252,G,T,0.006,0.677,0.1120,1.42e-09,37364,96.31%,OGFRL1,OGFRL1,6,71185549,71185548
4,,rs2980888,rs2980888,8,126507308,T,C,0.286,0.130,0.0072,4.21e-72,218595,99.95%,lnc-TRIB1-2;WASHC5,TRIB1,8,125495066,125495065
5,,rs4484649,rs4484649,8,10571491,C,A,0.419,0.045,0.0066,1.38e-11,218595,98.51%,SOX7;RP1L1;C8orf74,RP1L1;SOX7,8,10713981,10713980
6,,rs4841132,rs4841132,8,9183596,A,G,0.106,0.123,0.0105,6.62e-32,218595,98.79%,PPP1R3B,PPP1R3B;TNKS;MFHAS1,8,9326086,9326085


In [19]:
coloc.results.dir <- '/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/qtl.coloc/gwas/coloc.res/'
coloc.res <- read.table(paste0(coloc.results.dir, '241205_WE_Full_Coloc_Summary_Stats_PP4_0.8_r2_0.5.tsv'),
                        sep='\t', header=T)
dim(coloc.res)
head(coloc.res)

[1] 213  16

,nsnps,PP.H0.abf,PP.H1.abf,PP.H2.abf,PP.H3.abf,PP.H4.abf,cell,modality,feature,qtl.lead,qtl.pval,qtl.pos,gwas.lead,gwas.pval,qtl.pos.1,r2
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<chr>,<dbl>,<int>,<dbl>
1,2825,9.681856e-10,1.149327e-04,2.216742e-07,0.025340263,0.9745446,Bulk,ATAC,chr1-16189980-16190280,rs1497406,1.547193e-10,16178825,rs1497406,1.458e-08,16178825,1.000
2,2566,2.705990e-05,2.476040e-02,1.156288e-05,0.009614698,0.9655863,Bulk,ATAC,chr1-229256817-229257117,rs10916454,2.233151e-09,229253038,rs12041301,9.290e-06,229245010,0.816
3,5058,5.088451e-02,6.435270e-02,1.102903e-02,0.013087567,0.8606462,Bulk,ATAC,chr10-26710287-26710575,rs2477276,5.178316e-08,26666136,rs2477276,2.912e-05,26666136,1.000
4,5057,3.724166e-02,5.371677e-02,8.071817e-03,0.010752443,0.8902173,Bulk,ATAC,chr10-26711375-26711675,rs2477276,3.571726e-08,26666136,rs2477276,2.912e-05,26666136,1.000
5,2439,2.088562e-34,1.195832e-10,1.869171e-25,0.106127774,0.8938722,Bulk,ATAC,chr11-61820741-61821041,rs174576,2.043734e-17,61836038,rs174535,3.105e-15,61783884,0.911
6,2412,4.655157e-12,5.397353e-11,4.166160e-03,0.047355445,0.9484784,Bulk,ATAC,chr11-61832766-61833066,rs174576,4.284548e-07,61836038,rs174535,3.105e-15,61783884,0.911


In [20]:
sum(cred.set$CS.SNP %in% coloc.res$gwas.lead)

[1] 16

In [21]:
cell <- c('Hepatocytes')
atac.qtl.coloc.vars.res <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid')) {
    for (i in 1:22) {
        atac.qtl.coloc.vars.res <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/05_caQTLs/outs/',cell,'.cis_qtl_pairs.chr',i,'.parquet.sig.tsv')) %>%
            filter(variant_id %in% c(cred.set$CS.SNP)) %>%
            mutate(cell=cell) %>%
            rbind(atac.qtl.coloc.vars.res)
    }
}

dim(atac.qtl.coloc.vars.res)
head(atac.qtl.coloc.vars.res)

Rows: 0 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 6 Columns: 12
── Column specification ──────────────────────────────────────────────────────────────

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 3 Columns: 12
── Column specification ──────────────────────────────────────────────────────────────────────────────────

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 747 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 10 Columns: 12
─


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 37 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 39 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ

Rows: 4007 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1333 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message

...

Rows: 1518 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 755 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 581 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 10352 Columns: 1


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2138 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2337 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qt

[1] 1260   13

phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,pval_nominal_threshold,sig.qtl,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>
chr19-4327352-4327620,rs35013670,9675,9408,0.3779070,56,65,5.542695e-07,0.8183219,0.1437315,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs28654688,12821,12554,0.4418605,65,76,1.947807e-06,0.7966865,0.1490473,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs3810367,15497,15230,0.4476744,66,77,2.106467e-06,0.7794851,0.1464282,3.04817e-06,TRUE,Myeloid
chr19-4327651-4327951,rs35013670,9376,9077,0.3779070,56,65,2.155003e-06,0.8106108,0.1524574,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs28654688,12522,12223,0.4418605,65,76,8.085321e-07,0.8481727,0.1517498,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs12978346,13093,12794,0.3023256,45,52,1.143572e-06,0.7831656,0.1425668,3.29284e-06,TRUE,Myeloid


In [22]:
rna.qtl.coloc.vars.res <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid')) {
    rna.qtl.coloc.vars.res <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/04_eQTLs/outs/',cell,'.cis_qtl_pairs.chr.qvalue.sig.tsv')) %>%
        filter(variant_id %in% c(cred.set$CS.SNP)) %>%
        mutate(cell=cell) %>%
        rbind(rna.qtl.coloc.vars.res)
}

dim(rna.qtl.coloc.vars.res)
head(rna.qtl.coloc.vars.res)

Rows: 653 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (8): start_distance, af, ma_samples, ma_count, pval_nominal, slope, slop...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 8140 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (8): start_distance, af, ma_samples, ma_count, pval_nominal, slope, slop...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

[1] 52 12

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>
HSD17B13,rs10433879,-12898,0.2034884,30,35,4.227959e-06,0.7136446,0.13917176,TRUE,1.03562e-05,Myeloid
HLA-DQB1,rs968155,-255445,0.6453488,48,61,8.382722e-06,-0.5900655,0.11959907,TRUE,1.02602e-05,Myeloid
HLA-DQB1,rs9271406,-48572,0.5930232,54,70,8.116191e-08,-0.6510572,0.10469777,TRUE,1.02602e-05,Myeloid
EPHA2,rs11588341,15484,0.5232558,61,82,9.513265e-06,-0.3773269,0.07704365,TRUE,2.53574e-05,Hepatocytes
EPHA2,rs1497407,17041,0.4883721,60,84,8.349330e-06,-0.3804022,0.07708509,TRUE,2.53574e-05,Hepatocytes
EPHA2,rs7519043,17959,0.5174419,62,83,6.773970e-06,-0.3836847,0.07682361,TRUE,2.53574e-05,Hepatocytes


In [23]:
H3K27ac.qtl.coloc.vars.res <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid')) {
    H3K27ac.qtl.coloc.vars.res <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/06_H3K27acQTLs/outs/',cell,'.cis_qtl_pairs.chr.qvalue.sig.tsv')) %>%
        filter(variant_id %in% c(cred.set$CS.SNP)) %>%
        mutate(cell=cell) %>%
        rbind(H3K27ac.qtl.coloc.vars.res)
}

dim(H3K27ac.qtl.coloc.vars.res)
head(H3K27ac.qtl.coloc.vars.res)

Rows: 0 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (11): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (11): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1653 Columns: 12
── Column specification ───────────────────────────────────────────────────────────

[1] 101  13

phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>
chr8:8227666-8228665,rs2979172,225331,224333,0.6395349,47,62,1.630880e-09,-0.9632696,0.1324173,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs4841040,569350,568352,0.5872093,56,71,5.232810e-09,-0.9810939,0.1409638,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs6994038,575361,574363,0.5872093,56,71,2.781884e-08,-0.9362662,0.1438492,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs4841042,579445,578447,0.5930232,54,70,5.722191e-09,-0.9160417,0.1320748,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs7823757,585000,584002,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs60315134,585422,584424,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid


In [24]:
H3K27me3.qtl.coloc.vars.res <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid')) {
    H3K27me3.qtl.coloc.vars.res <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/07_H3K27me3QTLs/outs/',cell,'.cis_qtl_pairs.chr.qvalue.sig.tsv')) %>%
        filter(variant_id %in% c(cred.set$CS.SNP)) %>%
        mutate(cell=cell) %>%
        rbind(H3K27me3.qtl.coloc.vars.res)
}

dim(H3K27me3.qtl.coloc.vars.res)
head(H3K27me3.qtl.coloc.vars.res)

Rows: 0 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (11): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 54 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 11
── Column specification ──────────

[1] 14 13

phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>
chr6:32550868-32552367,rs9268839,-89874,-91372,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs13211921,-75810,-77308,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs9391879,-75474,-76972,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs12195589,-74062,-75560,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs28895253,-73736,-75234,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs28895257,-73532,-75030,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes


In [ ]:
# Quick look at overlap

In [25]:
length(unique(cred.set$LEAD.SNP))

[1] 77

In [26]:
sum(cred.set$CS.SNP %in% atac.qtl.coloc.vars.res$variant_id)
sum(cred.set$CS.SNP %in% rna.qtl.coloc.vars.res$variant_id)
sum(cred.set$CS.SNP %in% H3K27ac.qtl.coloc.vars.res$variant_id)
sum(cred.set$CS.SNP %in% H3K27me3.qtl.coloc.vars.res$variant_id)

sum(cred.set$CS.SNP %in% c(atac.qtl.coloc.vars.res$variant_id,rna.qtl.coloc.vars.res$variant_id,
                           H3K27ac.qtl.coloc.vars.res$variant_id,H3K27me3.qtl.coloc.vars.res$variant_id))

[1] 410

[1] 49

[1] 49

[1] 14

[1] 423

In [27]:
unique(atac.qtl.coloc.vars.res$cell)
unique(rna.qtl.coloc.vars.res$cell)
unique(H3K27ac.qtl.coloc.vars.res$cell)
unique(H3K27me3.qtl.coloc.vars.res$cell)

[1] "Myeloid"       "HSC"           "Hepatocytes"   "Endothelial"  
[5] "Cholangiocyte" "B"

[1] "Myeloid"     "Hepatocytes" "Endothelial"

[1] "Myeloid"     "Hepatocytes" "Endothelial"

[1] "Hepatocytes"

In [28]:
filter(atac.qtl.coloc.vars.res, cell==cell)

phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,pval_nominal_threshold,sig.qtl,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>
chr19-4327352-4327620,rs35013670,9675,9408,0.3779070,56,65,5.542695e-07,0.8183219,0.1437315,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs28654688,12821,12554,0.4418605,65,76,1.947807e-06,0.7966865,0.1490473,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs3810367,15497,15230,0.4476744,66,77,2.106467e-06,0.7794851,0.1464282,3.04817e-06,TRUE,Myeloid
chr19-4327651-4327951,rs35013670,9376,9077,0.3779070,56,65,2.155003e-06,0.8106108,0.1524574,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs28654688,12522,12223,0.4418605,65,76,8.085321e-07,0.8481727,0.1517498,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs12978346,13093,12794,0.3023256,45,52,1.143572e-06,0.7831656,0.1425668,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs3810368,15081,14782,0.2965116,45,51,1.676921e-06,0.7826868,0.1452934,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs3810367,15198,14899,0.4476744,66,77,1.988750e-06,0.8080804,0.1513433,3.29284e-06,TRUE,Myeloid
chr19-4328187-4328487,rs12978346,12557,12258,0.3023256,45,52,2.433819e-06,0.5466126,0.1034675,3.05865e-06,TRUE,Myeloid


In [52]:
cred.set.coloc <- cred.set

for(cell.type in unique(atac.qtl.coloc.vars.res$cell)) {
    cred.set.coloc[[paste0('ATAC.QTL.Intersect.',cell.type)]] <- FALSE
    cred.set.coloc[cred.set.coloc$CS.SNP %in% filter(atac.qtl.coloc.vars.res, cell==cell.type)$variant_id,][[paste0('ATAC.QTL.Intersect.',cell.type)]] <- TRUE
}

for(cell.type in unique(rna.qtl.coloc.vars.res$cell)) {
    cred.set.coloc[[paste0('RNA.QTL.Intersect.',cell.type)]] <- FALSE
    cred.set.coloc[cred.set.coloc$CS.SNP %in% filter(rna.qtl.coloc.vars.res, cell==cell.type)$variant_id,][[paste0('RNA.QTL.Intersect.',cell.type)]] <- TRUE
}

for(cell.type in unique(H3K27ac.qtl.coloc.vars.res$cell)) {
    cred.set.coloc[[paste0('H3K27ac.QTL.Intersect.',cell.type)]] <- FALSE
    cred.set.coloc[cred.set.coloc$CS.SNP %in% filter(H3K27ac.qtl.coloc.vars.res, cell==cell.type)$variant_id,][[paste0('H3K27ac.QTL.Intersect.',cell.type)]] <- TRUE
}

for(cell.type in unique(H3K27me3.qtl.coloc.vars.res$cell)) {
    cred.set.coloc[[paste0('H3K27me3.QTL.Intersect.',cell.type)]] <- FALSE
    cred.set.coloc[cred.set.coloc$CS.SNP %in% filter(H3K27me3.qtl.coloc.vars.res, cell==cell.type)$variant_id,][[paste0('H3K27me3.QTL.Intersect.',cell.type)]] <- TRUE
}

#cred.set.coloc

select(cred.set.coloc,LEAD.SNP, contains('QTL.Intersect')) %>%
    group_by(LEAD.SNP) %>%
    summarise_all(any) %>%
    select(-LEAD.SNP) %>%
    summarise_all(sum)

ATAC.QTL.Intersect.Myeloid,ATAC.QTL.Intersect.HSC,ATAC.QTL.Intersect.Hepatocytes,ATAC.QTL.Intersect.Endothelial,ATAC.QTL.Intersect.Cholangiocyte,ATAC.QTL.Intersect.B,RNA.QTL.Intersect.Myeloid,RNA.QTL.Intersect.Hepatocytes,RNA.QTL.Intersect.Endothelial,H3K27ac.QTL.Intersect.Myeloid,H3K27ac.QTL.Intersect.Hepatocytes,H3K27ac.QTL.Intersect.Endothelial,H3K27me3.QTL.Intersect.Hepatocytes
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
8,2,26,7,2,1,2,6,1,1,6,2,2


# Start by joining summary stats and credible sets

In [35]:
head(cred.set)

,CS.Type,LEAD.SNP,CS.SNP,Chr,Position,EA,NEA,EAF,Beta,SE,⋯,ATAC.QTL.Coloc.Endothelial,ATAC.QTL.Coloc.Cholangiocyte,ATAC.QTL.Coloc.B,RNA.QTL.Coloc.Myeloid,RNA.QTL.Coloc.Hepatocytes,RNA.QTL.Coloc.Endothelial,H3K27ac.QTL.Coloc.Myeloid,H3K27ac.QTL.Coloc.Hepatocytes,H3K27ac.QTL.Coloc.Endothelial,H3K27me3.QTL.Coloc.Hepatocytes
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
1,Single SNP,rs2642438,rs2642438,1,220970028,A,G,0.274,-0.075,0.0074,⋯,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
2,,rs6734238,rs6734238,2,113841030,G,A,0.407,-0.057,0.0064,⋯,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
3,,rs138033684,rs138033684,6,71895252,G,T,0.006,0.677,0.1120,⋯,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
4,,rs2980888,rs2980888,8,126507308,T,C,0.286,0.130,0.0072,⋯,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
5,,rs4484649,rs4484649,8,10571491,C,A,0.419,0.045,0.0066,⋯,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
6,,rs4841132,rs4841132,8,9183596,A,G,0.106,0.123,0.0105,⋯,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE


In [46]:
head(atac.qtl.coloc.vars.res)

phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,pval_nominal_threshold,sig.qtl,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>
chr19-4327352-4327620,rs35013670,9675,9408,0.3779070,56,65,5.542695e-07,0.8183219,0.1437315,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs28654688,12821,12554,0.4418605,65,76,1.947807e-06,0.7966865,0.1490473,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs3810367,15497,15230,0.4476744,66,77,2.106467e-06,0.7794851,0.1464282,3.04817e-06,TRUE,Myeloid
chr19-4327651-4327951,rs35013670,9376,9077,0.3779070,56,65,2.155003e-06,0.8106108,0.1524574,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs28654688,12522,12223,0.4418605,65,76,8.085321e-07,0.8481727,0.1517498,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs12978346,13093,12794,0.3023256,45,52,1.143572e-06,0.7831656,0.1425668,3.29284e-06,TRUE,Myeloid


In [49]:
inner_join(cred.set, atac.qtl.coloc.vars.res, join_by(CS.SNP==variant_id)) %>%
    select(LEAD.SNP, cell) %>% distinct() %>% group_by(cell) %>% summarise(count=n())

cell,count
<chr>,<int>
B,1
Cholangiocyte,2
Endothelial,7
HSC,2
Hepatocytes,26
Myeloid,8


In [ ]:
# That worked okay for caQTLs, let's merge all QTL sumamry stats

In [53]:
caQTL.cred.set.overlap.full <- filter(atac.qtl.coloc.vars.res, variant_id %in% cred.set$CS.SNP) %>%
    select(caQTL.Peak=phenotype_id, variant_id, qtl.cell=cell, qtp.pval_nominal=pval_nominal, sig.qtl, qtl.slope=slope, qtl.slope_se=slope_se) %>%
    distinct() %>%
    filter(!is.na(variant_id)) %>%
    left_join(select(cred.set, -Chr, -Position, -Nominated.Gene, -Biological.Prior.Gene, start.hg38, pos.hg38=end.hg38), by=join_by('variant_id'=='CS.SNP'))
   

length(unique(caQTL.cred.set.overlap.full$LEAD.SNP))
caQTL.cred.set.overlap.full


[1] 31

caQTL.Peak,variant_id,qtl.cell,qtp.pval_nominal,sig.qtl,qtl.slope,qtl.slope_se,CS.Type,LEAD.SNP,EA,NEA,EAF,Beta,SE,P,N,PP,chr.hg38,pos.hg38,start.hg38
<chr>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<int>
chr19-4327352-4327620,rs35013670,Myeloid,5.542695e-07,TRUE,0.8183219,0.1437315,,rs3810367,C,T,0.353,0.031,0.0066,3.15e-06,218595,0.51%,19,4337028,4337027
chr19-4327352-4327620,rs28654688,Myeloid,1.947807e-06,TRUE,0.7966865,0.1490473,,rs3810367,G,A,0.370,0.036,0.0067,4.74e-08,218595,21.09%,19,4340174,4340173
chr19-4327352-4327620,rs3810367,Myeloid,2.106467e-06,TRUE,0.7794851,0.1464282,,rs3810367,G,T,0.374,0.037,0.0066,3.77e-08,218595,38.43%,19,4342850,4342849
chr19-4327651-4327951,rs35013670,Myeloid,2.155003e-06,TRUE,0.8106108,0.1524574,,rs3810367,C,T,0.353,0.031,0.0066,3.15e-06,218595,0.51%,19,4337028,4337027
chr19-4327651-4327951,rs28654688,Myeloid,8.085321e-07,TRUE,0.8481727,0.1517498,,rs3810367,G,A,0.370,0.036,0.0067,4.74e-08,218595,21.09%,19,4340174,4340173
chr19-4327651-4327951,rs12978346,Myeloid,1.143572e-06,TRUE,0.7831656,0.1425668,,rs3810367,A,C,0.283,0.034,0.0072,3.25e-06,218595,0.55%,19,4340745,4340744
chr19-4327651-4327951,rs3810368,Myeloid,1.676921e-06,TRUE,0.7826868,0.1452934,,rs3810367,A,G,0.283,0.034,0.0073,3.16e-06,218595,0.53%,19,4342733,4342732
chr19-4327651-4327951,rs3810367,Myeloid,1.988750e-06,TRUE,0.8080804,0.1513433,,rs3810367,G,T,0.374,0.037,0.0066,3.77e-08,218595,38.43%,19,4342850,4342849
chr19-4328187-4328487,rs12978346,Myeloid,2.433819e-06,TRUE,0.5466126,0.1034675,,rs3810367,A,C,0.283,0.034,0.0072,3.25e-06,218595,0.55%,19,4340745,4340744


In [57]:
head(rna.qtl.coloc.vars.res)
head(atac.qtl.coloc.vars.res)
head(H3K27ac.qtl.coloc.vars.res)
head(H3K27me3.qtl.coloc.vars.res)

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>
HSD17B13,rs10433879,-12898,0.2034884,30,35,4.227959e-06,0.7136446,0.13917176,TRUE,1.03562e-05,Myeloid
HLA-DQB1,rs968155,-255445,0.6453488,48,61,8.382722e-06,-0.5900655,0.11959907,TRUE,1.02602e-05,Myeloid
HLA-DQB1,rs9271406,-48572,0.5930232,54,70,8.116191e-08,-0.6510572,0.10469777,TRUE,1.02602e-05,Myeloid
EPHA2,rs11588341,15484,0.5232558,61,82,9.513265e-06,-0.3773269,0.07704365,TRUE,2.53574e-05,Hepatocytes
EPHA2,rs1497407,17041,0.4883721,60,84,8.349330e-06,-0.3804022,0.07708509,TRUE,2.53574e-05,Hepatocytes
EPHA2,rs7519043,17959,0.5174419,62,83,6.773970e-06,-0.3836847,0.07682361,TRUE,2.53574e-05,Hepatocytes


phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,pval_nominal_threshold,sig.qtl,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>
chr19-4327352-4327620,rs35013670,9675,9408,0.3779070,56,65,5.542695e-07,0.8183219,0.1437315,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs28654688,12821,12554,0.4418605,65,76,1.947807e-06,0.7966865,0.1490473,3.04817e-06,TRUE,Myeloid
chr19-4327352-4327620,rs3810367,15497,15230,0.4476744,66,77,2.106467e-06,0.7794851,0.1464282,3.04817e-06,TRUE,Myeloid
chr19-4327651-4327951,rs35013670,9376,9077,0.3779070,56,65,2.155003e-06,0.8106108,0.1524574,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs28654688,12522,12223,0.4418605,65,76,8.085321e-07,0.8481727,0.1517498,3.29284e-06,TRUE,Myeloid
chr19-4327651-4327951,rs12978346,13093,12794,0.3023256,45,52,1.143572e-06,0.7831656,0.1425668,3.29284e-06,TRUE,Myeloid


phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>
chr8:8227666-8228665,rs2979172,225331,224333,0.6395349,47,62,1.630880e-09,-0.9632696,0.1324173,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs4841040,569350,568352,0.5872093,56,71,5.232810e-09,-0.9810939,0.1409638,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs6994038,575361,574363,0.5872093,56,71,2.781884e-08,-0.9362662,0.1438492,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs4841042,579445,578447,0.5930232,54,70,5.722191e-09,-0.9160417,0.1320748,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs7823757,585000,584002,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid
chr8:8227666-8228665,rs60315134,585422,584424,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid


phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>
chr6:32550868-32552367,rs9268839,-89874,-91372,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs13211921,-75810,-77308,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs9391879,-75474,-76972,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs12195589,-74062,-75560,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs28895253,-73736,-75234,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes
chr6:32550868-32552367,rs28895257,-73532,-75030,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes


In [68]:
# Fix columns to match
rna.qtl.coloc.vars.res.mod <- rna.qtl.coloc.vars.res %>% mutate(modality='RNA')
atac.qtl.coloc.vars.res.mod <- select(atac.qtl.coloc.vars.res, -end_distance) %>%
    relocate(pval_nominal_threshold, .after = sig.qtl) %>% mutate(modality='ATAC')
H3K27ac.qtl.coloc.vars.res.mod <- select(H3K27ac.qtl.coloc.vars.res, -end_distance) %>% mutate(modality='H3K27ac')
H3K27me3.qtl.coloc.vars.res.mod <- select(H3K27me3.qtl.coloc.vars.res, -end_distance) %>% mutate(modality='H3K27me3')

In [69]:
head(rna.qtl.coloc.vars.res.mod)
head(atac.qtl.coloc.vars.res.mod)
head(H3K27ac.qtl.coloc.vars.res.mod)
head(H3K27me3.qtl.coloc.vars.res.mod)

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
HSD17B13,rs10433879,-12898,0.2034884,30,35,4.227959e-06,0.7136446,0.13917176,TRUE,1.03562e-05,Myeloid,RNA
HLA-DQB1,rs968155,-255445,0.6453488,48,61,8.382722e-06,-0.5900655,0.11959907,TRUE,1.02602e-05,Myeloid,RNA
HLA-DQB1,rs9271406,-48572,0.5930232,54,70,8.116191e-08,-0.6510572,0.10469777,TRUE,1.02602e-05,Myeloid,RNA
EPHA2,rs11588341,15484,0.5232558,61,82,9.513265e-06,-0.3773269,0.07704365,TRUE,2.53574e-05,Hepatocytes,RNA
EPHA2,rs1497407,17041,0.4883721,60,84,8.349330e-06,-0.3804022,0.07708509,TRUE,2.53574e-05,Hepatocytes,RNA
EPHA2,rs7519043,17959,0.5174419,62,83,6.773970e-06,-0.3836847,0.07682361,TRUE,2.53574e-05,Hepatocytes,RNA


phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
chr19-4327352-4327620,rs35013670,9675,0.3779070,56,65,5.542695e-07,0.8183219,0.1437315,TRUE,3.04817e-06,Myeloid,ATAC
chr19-4327352-4327620,rs28654688,12821,0.4418605,65,76,1.947807e-06,0.7966865,0.1490473,TRUE,3.04817e-06,Myeloid,ATAC
chr19-4327352-4327620,rs3810367,15497,0.4476744,66,77,2.106467e-06,0.7794851,0.1464282,TRUE,3.04817e-06,Myeloid,ATAC
chr19-4327651-4327951,rs35013670,9376,0.3779070,56,65,2.155003e-06,0.8106108,0.1524574,TRUE,3.29284e-06,Myeloid,ATAC
chr19-4327651-4327951,rs28654688,12522,0.4418605,65,76,8.085321e-07,0.8481727,0.1517498,TRUE,3.29284e-06,Myeloid,ATAC
chr19-4327651-4327951,rs12978346,13093,0.3023256,45,52,1.143572e-06,0.7831656,0.1425668,TRUE,3.29284e-06,Myeloid,ATAC


phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
chr8:8227666-8228665,rs2979172,225331,0.6395349,47,62,1.630880e-09,-0.9632696,0.1324173,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs4841040,569350,0.5872093,56,71,5.232810e-09,-0.9810939,0.1409638,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs6994038,575361,0.5872093,56,71,2.781884e-08,-0.9362662,0.1438492,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs4841042,579445,0.5930232,54,70,5.722191e-09,-0.9160417,0.1320748,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs7823757,585000,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs60315134,585422,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid,H3K27ac


phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
chr6:32550868-32552367,rs9268839,-89874,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs13211921,-75810,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs9391879,-75474,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs12195589,-74062,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs28895253,-73736,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs28895257,-73532,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3


In [70]:
# Do they match? If so bind them
sum(length(colnames(rna.qtl.coloc.vars.res.mod)) == length(colnames(atac.qtl.coloc.vars.res.mod)))
sum(length(colnames(rna.qtl.coloc.vars.res.mod)) == length(colnames(H3K27ac.qtl.coloc.vars.res.mod)))
sum(length(colnames(rna.qtl.coloc.vars.res.mod)) == length(colnames(H3K27me3.qtl.coloc.vars.res.mod)))

sum(colnames(rna.qtl.coloc.vars.res.mod) == colnames(atac.qtl.coloc.vars.res.mod))
sum(colnames(rna.qtl.coloc.vars.res.mod) == colnames(H3K27ac.qtl.coloc.vars.res.mod))
sum(colnames(rna.qtl.coloc.vars.res.mod) == colnames(H3K27me3.qtl.coloc.vars.res.mod))

[1] 1

[1] 1

[1] 1

[1] 13

[1] 13

[1] 13

In [71]:
qtl.cs.vars <- rbind(rna.qtl.coloc.vars.res.mod, atac.qtl.coloc.vars.res.mod) %>%
    rbind(H3K27ac.qtl.coloc.vars.res.mod) %>%
    rbind(H3K27me3.qtl.coloc.vars.res.mod)

dim(qtl.cs.vars)
head(qtl.cs.vars)

[1] 1427   13

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
HSD17B13,rs10433879,-12898,0.2034884,30,35,4.227959e-06,0.7136446,0.13917176,TRUE,1.03562e-05,Myeloid,RNA
HLA-DQB1,rs968155,-255445,0.6453488,48,61,8.382722e-06,-0.5900655,0.11959907,TRUE,1.02602e-05,Myeloid,RNA
HLA-DQB1,rs9271406,-48572,0.5930232,54,70,8.116191e-08,-0.6510572,0.10469777,TRUE,1.02602e-05,Myeloid,RNA
EPHA2,rs11588341,15484,0.5232558,61,82,9.513265e-06,-0.3773269,0.07704365,TRUE,2.53574e-05,Hepatocytes,RNA
EPHA2,rs1497407,17041,0.4883721,60,84,8.349330e-06,-0.3804022,0.07708509,TRUE,2.53574e-05,Hepatocytes,RNA
EPHA2,rs7519043,17959,0.5174419,62,83,6.773970e-06,-0.3836847,0.07682361,TRUE,2.53574e-05,Hepatocytes,RNA


In [76]:
qtl.cred.set <- select(qtl.cs.vars, qtl.feature=phenotype_id, variant_id, qtl.cell=cell, qtp.pval_nominal=pval_nominal, sig.qtl, qtl.slope=slope, qtl.slope_se=slope_se, qtl.modality=modality) %>%
    distinct() %>%
    filter(!is.na(variant_id)) %>%
    left_join(select(cred.set, -Chr, -Position, -Nominated.Gene, -Biological.Prior.Gene, start.hg38, pos.hg38=end.hg38), by=join_by('variant_id'=='CS.SNP'))
   

length(unique(qtl.cred.set$LEAD.SNP))

select(qtl.cred.set, LEAD.SNP, qtl.cell, qtl.modality) %>% distinct() %>% group_by(qtl.cell,qtl.modality) %>% summarise(count=n())
select(qtl.cred.set, LEAD.SNP, qtl.cell) %>% distinct() %>% group_by(qtl.cell) %>% summarise(count=n())

qtl.cred.set



[1] 33

`summarise()` has grouped output by 'qtl.cell'. You can override using the `.groups` argument.


qtl.cell,qtl.modality,count
<chr>,<chr>,<int>
B,ATAC,1
Cholangiocyte,ATAC,2
Endothelial,ATAC,7
Endothelial,H3K27ac,2
Endothelial,RNA,1
HSC,ATAC,2
Hepatocytes,ATAC,26
Hepatocytes,H3K27ac,6
Hepatocytes,H3K27me3,2


qtl.cell,count
<chr>,<int>
B,1
Cholangiocyte,2
Endothelial,7
HSC,2
Hepatocytes,28
Myeloid,9


qtl.feature,variant_id,qtl.cell,qtp.pval_nominal,sig.qtl,qtl.slope,qtl.slope_se,qtl.modality,CS.Type,LEAD.SNP,⋯,NEA,EAF,Beta,SE,P,N,PP,chr.hg38,pos.hg38,start.hg38
<chr>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<int>,<int>,<int>
HSD17B13,rs10433879,Myeloid,4.227959e-06,TRUE,0.7136446,0.13917176,RNA,,rs10433937,⋯,G,0.255,-0.081,0.0077,5.59e-26,218595,15.93%,4,87309988,87309987
HLA-DQB1,rs968155,Myeloid,8.382722e-06,TRUE,-0.5900655,0.11959907,RNA,,rs686250,⋯,T,0.462,-0.036,0.0063,1.16e-08,218595,5.82%,6,32412938,32412937
HLA-DQB1,rs9271406,Myeloid,8.116191e-08,TRUE,-0.6510572,0.10469777,RNA,,rs686250,⋯,A,0.454,0.034,0.0064,1.25e-07,218595,0.50%,6,32619811,32619810
EPHA2,rs11588341,Hepatocytes,9.513265e-06,TRUE,-0.3773269,0.07704365,RNA,,rs36086195,⋯,G,0.483,-0.039,0.0067,5.65e-09,218595,3.41%,1,16171553,16171552
EPHA2,rs1497407,Hepatocytes,8.349330e-06,TRUE,-0.3804022,0.07708509,RNA,,rs36086195,⋯,C,0.483,-0.042,0.0068,6.50e-10,218595,29.37%,1,16173110,16173109
EPHA2,rs7519043,Hepatocytes,6.773970e-06,TRUE,-0.3836847,0.07682361,RNA,,rs36086195,⋯,T,0.485,-0.039,0.0067,4.52e-09,218595,4.06%,1,16174028,16174027
EPHA2,rs7538833,Hepatocytes,1.494710e-06,TRUE,-0.3871493,0.07144354,RNA,,rs36086195,⋯,C,0.360,-0.039,0.0068,6.68e-09,218595,2.29%,1,16177886,16177885
EPHA2,rs1497406,Hepatocytes,1.063339e-05,TRUE,-0.3844107,0.07900385,RNA,,rs36086195,⋯,G,0.475,-0.039,0.0066,3.56e-09,218595,5.18%,1,16178825,16178824
EPHA2,rs4661718,Hepatocytes,3.125939e-06,TRUE,-0.3672506,0.07045192,RNA,,rs36086195,⋯,T,0.364,-0.039,0.0066,4.96e-09,218595,3.63%,1,16179413,16179412


# Add if it is a coloc example

In [80]:
coloc.select <- coloc.res %>%
    select(PP.H4.abf, qtl.cell=cell, qtl.modality=modality, qtl.feature=feature, gwas.lead, gwas.pval, qtl.pos1=qtl.pos, r2)

dim(coloc.select)
head(coloc.select)

[1] 213   8

,PP.H4.abf,qtl.cell,qtl.modality,qtl.feature,gwas.lead,gwas.pval,qtl.pos1,r2
,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>
1,0.9745446,Bulk,ATAC,chr1-16189980-16190280,rs1497406,1.458e-08,16178825,1.000
2,0.9655863,Bulk,ATAC,chr1-229256817-229257117,rs12041301,9.290e-06,229253038,0.816
3,0.8606462,Bulk,ATAC,chr10-26710287-26710575,rs2477276,2.912e-05,26666136,1.000
4,0.8902173,Bulk,ATAC,chr10-26711375-26711675,rs2477276,2.912e-05,26666136,1.000
5,0.8938722,Bulk,ATAC,chr11-61820741-61821041,rs174535,3.105e-15,61836038,0.911
6,0.9484784,Bulk,ATAC,chr11-61832766-61833066,rs174535,3.105e-15,61836038,0.911


In [85]:
qtl.cred.set.coloc <- left_join(qtl.cred.set, coloc.select) %>%
    mutate(is.coloc=!is.na(PP.H4.abf))

dim(qtl.cred.set.coloc)
head(qtl.cred.set.coloc)

select(qtl.cred.set.coloc, LEAD.SNP, qtl.cell, qtl.modality) %>% distinct() %>% group_by(qtl.cell,qtl.modality) %>% summarise(count=n())
select(qtl.cred.set.coloc, LEAD.SNP, qtl.cell) %>% distinct() %>% group_by(qtl.cell) %>% summarise(count=n())

select(qtl.cred.set.coloc, LEAD.SNP, qtl.cell, qtl.modality, is.coloc) %>% distinct() %>% group_by(qtl.cell,qtl.modality, is.coloc) %>% summarise(count=n())
select(qtl.cred.set.coloc, LEAD.SNP, qtl.cell, is.coloc) %>% distinct() %>% group_by(qtl.cell, is.coloc) %>% summarise(count=n())

Joining with `by = join_by(qtl.feature, qtl.cell, qtl.modality)`


[1] 1427   27

qtl.feature,variant_id,qtl.cell,qtp.pval_nominal,sig.qtl,qtl.slope,qtl.slope_se,qtl.modality,CS.Type,LEAD.SNP,⋯,PP,chr.hg38,pos.hg38,start.hg38,PP.H4.abf,gwas.lead,gwas.pval,qtl.pos1,r2,is.coloc
<chr>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<int>,<int>,<dbl>,<chr>,<dbl>,<int>,<dbl>,<lgl>
HSD17B13,rs10433879,Myeloid,4.227959e-06,TRUE,0.7136446,0.13917176,RNA,,rs10433937,⋯,15.93%,4,87309988,87309987,NA,NA,NA,NA,NA,FALSE
HLA-DQB1,rs968155,Myeloid,8.382722e-06,TRUE,-0.5900655,0.11959907,RNA,,rs686250,⋯,5.82%,6,32412938,32412937,NA,NA,NA,NA,NA,FALSE
HLA-DQB1,rs9271406,Myeloid,8.116191e-08,TRUE,-0.6510572,0.10469777,RNA,,rs686250,⋯,0.50%,6,32619811,32619810,NA,NA,NA,NA,NA,FALSE
EPHA2,rs11588341,Hepatocytes,9.513265e-06,TRUE,-0.3773269,0.07704365,RNA,,rs36086195,⋯,3.41%,1,16171553,16171552,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE
EPHA2,rs1497407,Hepatocytes,8.349330e-06,TRUE,-0.3804022,0.07708509,RNA,,rs36086195,⋯,29.37%,1,16173110,16173109,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE
EPHA2,rs7519043,Hepatocytes,6.773970e-06,TRUE,-0.3836847,0.07682361,RNA,,rs36086195,⋯,4.06%,1,16174028,16174027,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE


`summarise()` has grouped output by 'qtl.cell'. You can override using the `.groups` argument.


qtl.cell,qtl.modality,count
<chr>,<chr>,<int>
B,ATAC,1
Cholangiocyte,ATAC,2
Endothelial,ATAC,7
Endothelial,H3K27ac,2
Endothelial,RNA,1
HSC,ATAC,2
Hepatocytes,ATAC,26
Hepatocytes,H3K27ac,6
Hepatocytes,H3K27me3,2


qtl.cell,count
<chr>,<int>
B,1
Cholangiocyte,2
Endothelial,7
HSC,2
Hepatocytes,28
Myeloid,9


`summarise()` has grouped output by 'qtl.cell', 'qtl.modality'. You can override using the `.groups` argument.


qtl.cell,qtl.modality,is.coloc,count
<chr>,<chr>,<lgl>,<int>
B,ATAC,FALSE,1
Cholangiocyte,ATAC,FALSE,2
Endothelial,ATAC,FALSE,7
Endothelial,ATAC,TRUE,1
Endothelial,H3K27ac,FALSE,2
Endothelial,H3K27ac,TRUE,1
Endothelial,RNA,TRUE,1
HSC,ATAC,FALSE,2
Hepatocytes,ATAC,FALSE,18


`summarise()` has grouped output by 'qtl.cell'. You can override using the `.groups` argument.


qtl.cell,is.coloc,count
<chr>,<lgl>,<int>
B,FALSE,1
Cholangiocyte,FALSE,2
Endothelial,FALSE,7
Endothelial,TRUE,1
HSC,FALSE,2
Hepatocytes,FALSE,20
Hepatocytes,TRUE,17
Myeloid,FALSE,9


In [87]:
write.table(qtl.cred.set.coloc, '/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/250325_WE_All_Mod_QTLs_cred_set_coloc_join.tsv', 
            sep='\t', col.names=T, row.names=F, quote=F)

In [88]:
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,900642,48.1,1417395,75.7,1417395,75.7
Vcells,2046011,15.7,8388608,64.0,4770614,36.4


# Overlapping cRE or Active state

In [97]:
in.peak.df <- data.frame()
in.active.df <- data.frame()


for (cell in unique(qtl.cred.set.coloc$qtl.cell)) {
    in.peak.df <- read.table(paste0('/nfs/lab/tscc/welison/FNIH.Liver/GWAS/peak.overlap/',cell,
                '.peak.MVP.NAFLD.TRANS.MVP.2021.credset.hg38.bed')) %>%
        select(variant_id=V4, V11) %>%
        filter(V11!=-1) %>%
        select(-V11) %>%
        mutate(qtl.cell=cell, is.in.peak=TRUE) %>%
        rbind(in.peak.df)
    
    in.active.df <- read.table(paste0('/nfs/lab/tscc/welison/FNIH.Liver/GWAS/peak.overlap/',cell,
                      '.states.MVP.NAFLD.TRANS.MVP.2021.credset.hg38.bed')) %>%
        select(variant_id=V4, V13) %>%
        filter(V13=='E5') %>%
        select(-V13) %>%
        mutate(qtl.cell=cell, is.in.active=TRUE) %>%
        rbind(in.active.df)
}

dim(in.peak.df)
group_by(in.peak.df, qtl.cell) %>% summarise(count=n())
dim(in.active.df)
group_by(in.active.df, qtl.cell) %>% summarise(count=n())

[1] 466   3

qtl.cell,count
<chr>,<int>
B,42
Cholangiocyte,68
Endothelial,73
HSC,79
Hepatocytes,115
Myeloid,89


[1] 1133    3

qtl.cell,count
<chr>,<int>
B,64
Cholangiocyte,141
Endothelial,167
HSC,128
Hepatocytes,397
Myeloid,236


In [103]:
qtl.cred.set.coloc.peak.active <- left_join(qtl.cred.set.coloc, in.peak.df, ) %>%
    left_join(in.active.df) %>%
    mutate(is.in.peak=ifelse(is.na(is.in.peak), FALSE, is.in.peak),
          is.in.active=ifelse(is.na(is.in.active), FALSE, is.in.active))

dim(qtl.cred.set.coloc.peak.active)
head(qtl.cred.set.coloc.peak.active)

Joining with `by = join_by(variant_id, qtl.cell)`
Joining with `by = join_by(variant_id, qtl.cell)`


[1] 1427   29

qtl.feature,variant_id,qtl.cell,qtp.pval_nominal,sig.qtl,qtl.slope,qtl.slope_se,qtl.modality,CS.Type,LEAD.SNP,⋯,pos.hg38,start.hg38,PP.H4.abf,gwas.lead,gwas.pval,qtl.pos1,r2,is.coloc,is.in.peak,is.in.active
<chr>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<int>,<int>,<dbl>,<chr>,<dbl>,<int>,<dbl>,<lgl>,<lgl>,<lgl>
HSD17B13,rs10433879,Myeloid,4.227959e-06,TRUE,0.7136446,0.13917176,RNA,,rs10433937,⋯,87309988,87309987,NA,NA,NA,NA,NA,FALSE,FALSE,FALSE
HLA-DQB1,rs968155,Myeloid,8.382722e-06,TRUE,-0.5900655,0.11959907,RNA,,rs686250,⋯,32412938,32412937,NA,NA,NA,NA,NA,FALSE,FALSE,FALSE
HLA-DQB1,rs9271406,Myeloid,8.116191e-08,TRUE,-0.6510572,0.10469777,RNA,,rs686250,⋯,32619811,32619810,NA,NA,NA,NA,NA,FALSE,FALSE,FALSE
EPHA2,rs11588341,Hepatocytes,9.513265e-06,TRUE,-0.3773269,0.07704365,RNA,,rs36086195,⋯,16171553,16171552,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE,FALSE,TRUE
EPHA2,rs1497407,Hepatocytes,8.349330e-06,TRUE,-0.3804022,0.07708509,RNA,,rs36086195,⋯,16173110,16173109,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE,FALSE,TRUE
EPHA2,rs7519043,Hepatocytes,6.773970e-06,TRUE,-0.3836847,0.07682361,RNA,,rs36086195,⋯,16174028,16174027,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE,FALSE,TRUE


In [105]:
select(qtl.cred.set.coloc.peak.active, LEAD.SNP, qtl.cell, is.in.peak) %>% distinct() %>% group_by(qtl.cell,is.in.peak) %>% summarise(count=n())
select(qtl.cred.set.coloc.peak.active, LEAD.SNP, qtl.cell, is.in.active) %>% distinct() %>% group_by(qtl.cell,is.in.active) %>% summarise(count=n())
select(qtl.cred.set.coloc.peak.active, LEAD.SNP, qtl.cell) %>% distinct() %>% group_by(qtl.cell) %>% summarise(count=n())

select(qtl.cred.set.coloc.peak.active, LEAD.SNP, is.in.peak) %>% distinct() %>% group_by(is.in.peak) %>% summarise(count=n())
select(qtl.cred.set.coloc.peak.active, LEAD.SNP, is.in.active) %>% distinct() %>% group_by(is.in.active) %>% summarise(count=n())


`summarise()` has grouped output by 'qtl.cell'. You can override using the `.groups` argument.


qtl.cell,is.in.peak,count
<chr>,<lgl>,<int>
B,FALSE,1
B,TRUE,1
Cholangiocyte,FALSE,2
Cholangiocyte,TRUE,2
Endothelial,FALSE,7
Endothelial,TRUE,3
HSC,FALSE,2
HSC,TRUE,1
Hepatocytes,FALSE,25


`summarise()` has grouped output by 'qtl.cell'. You can override using the `.groups` argument.


qtl.cell,is.in.active,count
<chr>,<lgl>,<int>
B,FALSE,1
Cholangiocyte,FALSE,2
Cholangiocyte,TRUE,1
Endothelial,FALSE,6
Endothelial,TRUE,4
HSC,FALSE,2
Hepatocytes,FALSE,17
Hepatocytes,TRUE,23
Myeloid,FALSE,9


qtl.cell,count
<chr>,<int>
B,1
Cholangiocyte,2
Endothelial,7
HSC,2
Hepatocytes,28
Myeloid,9


is.in.peak,count
<lgl>,<int>
FALSE,31
TRUE,20


is.in.active,count
<lgl>,<int>
FALSE,24
TRUE,26


In [106]:
write.table(qtl.cred.set.coloc.peak.active, '/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/250325_WE_All_Mod_QTLs_cred_set_coloc_join_peak_state_overlap.tsv', 
            sep='\t', col.names=T, row.names=F, quote=F)

# Add chromBPNet

In [109]:
chrombp <- read.table('/nfs/lab/tscc/welison/FNIH.Liver/13_chromBPNet_Peaks/variant_prediction/Control_snps_snp_scores.tsv', header=T)
dim(chrombp) 
head(chrombp)


[1] 1084    8

,CHR,POS0,REF,ALT,META_DATA,log_counts_diff,log_probs_diff_abs_sum,probs_jsd_diff
,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1,220796685,A,G,0.9686,0.197049620,104.89130,0.047965051
2,chr2,113083452,G,A,0.9904,-0.047235966,-28.07371,-0.016617813
3,chr6,71185548,G,T,0.9631,-0.007250309,-10.40595,-0.006723976
4,chr8,125495065,T,C,0.9995,0.107834340,35.40631,0.017021200
5,chr8,10713980,C,A,0.9851,0.169893740,44.13598,0.020204187
6,chr8,9326085,A,G,0.9879,0.204382180,71.86758,0.034811529


In [121]:
Hep.cs <- filter(qtl.cred.set.coloc.peak.active, qtl.cell=='Hepatocytes') %>%
    select(LEAD.SNP) %>% unlist() %>% unique()

length(Hep.cs)
head(Hep.cs)

[1] 28

[1] "rs36086195" "rs11683367" "rs10433937" "rs4841132"  "rs10883451"
[6] "rs1547014"

In [129]:
chrombp.good.effect <- inner_join(cred.set, chrombp, join_by(start.hg38 == POS0)) %>%
    select(-Nominated.Gene, -Biological.Prior.Gene, -chr.hg38, -end.hg38, 
           -start.hg38, -CHR, -META_DATA, -REF, -ALT) %>%
    filter(abs(log_counts_diff) > 0.025) 

filter(chrombp.good.effect, LEAD.SNP %in% Hep.cs) %>% select(LEAD.SNP) %>% distinct()
dim(chrombp.good.effect)
head(chrombp.good.effect)

LEAD.SNP
<chr>
rs4484649
rs4841132
rs36086195
rs11683409
rs2943652
rs6543007
rs4683438
rs7653249
rs10433937


[1] 260  16

,CS.Type,LEAD.SNP,CS.SNP,Chr,Position,EA,NEA,EAF,Beta,SE,P,N,PP,log_counts_diff,log_probs_diff_abs_sum,probs_jsd_diff
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<dbl>
1,Single SNP,rs2642438,rs2642438,1,220970028,A,G,0.274,-0.075,0.0074,6.65e-24,218595,96.86%,0.19704962,104.89130,0.04796505
2,,rs6734238,rs6734238,2,113841030,G,A,0.407,-0.057,0.0064,4.94e-19,218595,99.04%,-0.04723597,-28.07371,-0.01661781
3,,rs2980888,rs2980888,8,126507308,T,C,0.286,0.130,0.0072,4.21e-72,218595,99.95%,0.10783434,35.40631,0.01702120
4,,rs4484649,rs4484649,8,10571491,C,A,0.419,0.045,0.0066,1.38e-11,218595,98.51%,0.16989374,44.13598,0.02020419
5,,rs4841132,rs4841132,8,9183596,A,G,0.106,0.123,0.0105,6.62e-32,218595,98.79%,0.20438218,71.86758,0.03481153
6,,rs11601507,rs11601507,11,5701074,A,C,0.072,0.099,0.0128,1.53e-14,218595,100.00%,0.06414437,28.70793,0.01305975


In [133]:
colnames(chrombp.good.effect)

[1] "CS.Type"                "LEAD.SNP"               "CS.SNP"                
 [4] "Chr"                    "Position"               "EA"                    
 [7] "NEA"                    "EAF"                    "Beta"                  
[10] "SE"                     "P"                      "N"                     
[13] "PP"                     "log_counts_diff"        "log_probs_diff_abs_sum"
[16] "probs_jsd_diff"

In [135]:
chrombp.good.effect.to.join <- select(chrombp.good.effect,CS.SNP, chrombp_log_counts_diff=log_counts_diff, 
                                     chrombp_log_probs_diff_abs_sum=log_probs_diff_abs_sum,
                                     chrombp_probs_jsd_diff=probs_jsd_diff)

qtl.cred.set.coloc.peak.active_chrombp <- left_join(qtl.cred.set.coloc.peak.active, chrombp.good.effect.to.join, join_by(variant_id==CS.SNP))

dim(qtl.cred.set.coloc.peak.active)
head(qtl.cred.set.coloc.peak.active)

[1] 1427   29

qtl.feature,variant_id,qtl.cell,qtp.pval_nominal,sig.qtl,qtl.slope,qtl.slope_se,qtl.modality,CS.Type,LEAD.SNP,⋯,pos.hg38,start.hg38,PP.H4.abf,gwas.lead,gwas.pval,qtl.pos1,r2,is.coloc,is.in.peak,is.in.active
<chr>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<int>,<int>,<dbl>,<chr>,<dbl>,<int>,<dbl>,<lgl>,<lgl>,<lgl>
HSD17B13,rs10433879,Myeloid,4.227959e-06,TRUE,0.7136446,0.13917176,RNA,,rs10433937,⋯,87309988,87309987,NA,NA,NA,NA,NA,FALSE,FALSE,FALSE
HLA-DQB1,rs968155,Myeloid,8.382722e-06,TRUE,-0.5900655,0.11959907,RNA,,rs686250,⋯,32412938,32412937,NA,NA,NA,NA,NA,FALSE,FALSE,FALSE
HLA-DQB1,rs9271406,Myeloid,8.116191e-08,TRUE,-0.6510572,0.10469777,RNA,,rs686250,⋯,32619811,32619810,NA,NA,NA,NA,NA,FALSE,FALSE,FALSE
EPHA2,rs11588341,Hepatocytes,9.513265e-06,TRUE,-0.3773269,0.07704365,RNA,,rs36086195,⋯,16171553,16171552,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE,FALSE,TRUE
EPHA2,rs1497407,Hepatocytes,8.349330e-06,TRUE,-0.3804022,0.07708509,RNA,,rs36086195,⋯,16173110,16173109,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE,FALSE,TRUE
EPHA2,rs7519043,Hepatocytes,6.773970e-06,TRUE,-0.3836847,0.07682361,RNA,,rs36086195,⋯,16174028,16174027,0.8909272,rs1497406,1.458e-08,16177886,0.824,TRUE,FALSE,TRUE


In [136]:
write.table(qtl.cred.set.coloc.peak.active, '/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/250325_WE_All_Mod_QTLs_cred_set_coloc_join_peak_state_overlap_chromBP.tsv', 
            sep='\t', col.names=T, row.names=F, quote=F)